# Notebook 03: RAG Pipeline

Notebook ini mendemonstrasikan proses loading dokumen dari knowledge base, chunking, embedding generation menggunakan `multilingual-e5-small`, dan semantic search dengan FAISS.

In [1]:
import sys
import os
# Menambahkan path supaya bisa import src
sys.path.append(os.path.abspath('..'))

from src.rag_service import RAGService
import pandas as pd
from groq import Groq
import os
from dotenv import load_dotenv

## 1. Load Document & Inisialisasi RAG Service

In [2]:
rag = RAGService(index_dir='../faiss_index')
docs = rag.load_knowledge_base(kb_dir='../knowledge_base')
print(f'Total dokumen termuat: {len(docs)}')

if docs:
    print('\nContoh Metadata Dokumen 1:')
    print(docs[0].metadata)
    print('\nContoh Konten Dokumen 1:')
    print(docs[0].page_content[:200] + '...')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Total dokumen termuat: 14

Contoh Metadata Dokumen 1:
{'document_id': 'A002', 'type': 'addon', 'category': 'beverage', 'price': 5000, 'active': True, 'source': 'beverages.md'}

Contoh Konten Dokumen 1:
# 🥤 Add-ons - Minuman Tambahan

Lengkapi pesanan nasi kotak Anda dengan minuman segar pilihan kami!

## Daftar Minuman Tersedia

### Minuman Ringan (Rp 5.000)

**Freshea Pouch** - Rp 5.000  
Minuman k...


## 2. Chunking

In [3]:
chunked_docs = rag.chunk_documents(docs)
print(f'Total chunks setelah proses chunking: {len(chunked_docs)}')

if chunked_docs:
    print('\nContoh Chunk 1:')
    print(chunked_docs[0].page_content)

Total chunks setelah proses chunking: 20

Contoh Chunk 1:
# 🥤 Add-ons - Minuman Tambahan

Lengkapi pesanan nasi kotak Anda dengan minuman segar pilihan kami!

## Daftar Minuman Tersedia

### Minuman Ringan (Rp 5.000)

**Freshea Pouch** - Rp 5.000  
Minuman kesehatan dengan ekstrak buah-buahan segar. Cocok untuk menu sehat.

**Teh Kotak Sosro** - Rp 5.000  
Teh premium dari Sosro dalam kemasan praktis. Pilihan klasik yang disukai banyak orang.

**Es Teh** - Rp 5.000  
Es teh segar dengan rasa manis yang pas. Sempurna untuk minuman dingin di siang hari.

---

### Minuman Sedang (Rp 7.000)

**Es Jeruk** - Rp 7.000  
Jeruk segar yang diperas langsung dengan es batu. Menyegarkan dan alami.


## 3. Generate Embedding & Build FAISS Index

In [4]:
# Ini akan mendownload model jika belum ada (sekitar 470MB)
# Proses embedding mungkin memakan waktu beberapa detik
rag.build_index(chunked_docs)
rag.save_index()

Menghasilkan embedding untuk 20 chunks...
Berhasil membuat index FAISS dengan 20 vektor.
Index berhasil disimpan di ../faiss_index/


## 4. Semantic Search (Retrieval)

In [5]:
queries = [
    'Berapa harga nasi kotak minibox?',
    'Apakah bisa request custom menu untuk alergi seafood?',
    'Berapa lama maksimal pemesanan sebelum hari H?'
]

for query in queries:
    results = rag.search(query, top_k=2)
    print(f'Query: "{query}"')
    for i, res in enumerate(results, 1):
        print(f'--- Hasil {i} (Source: {res.metadata.get("source", "")}, Score: {res.metadata.get("relevance_score", 0)}) ---')
        print(res.page_content)
    print('\n' + '='*50 + '\n')


Query: "Berapa harga nasi kotak minibox?"
--- Hasil 1 (Source: Paket_Minibox.md, Score: 0.9142) ---
# Nasi Kotak Minibox

**Harga:** Rp 17.000 / box  
**Minimum Order:** 20 box  
**Cocok untuk:** Gathering, Office Break, Casual Event

## Menu Utama
- 1 potong ayam broiler reguler
- 1 nasi putih
- Sambal & lalapan

## Deskripsi
Nasi Kotak Minibox adalah pilihan terjangkau dari Ayam Bakar Pak D. Dengan daging ayam broiler berkualitas yang dibakar sempurna, menghadirkan cita rasa autentik Ayam Bakar Pak D dengan harga ekonomis. Cocok untuk kebutuhan sehari-hari atau acara santai.
--- Hasil 2 (Source: snack_box.md, Score: 0.895) ---
# Snack Box Tambahan

Selain hidangan utama nasi kotak, kami juga menyediakan opsi **Snack Box** yang cocok dihidangkan sebelum makan siang atau saat coffee break (co-break) acara Anda.

**Snack Box Standar - Rp 10.000 / box**
Isi:
- 1 Roti Manis / Lemper Ayam
- 1 Kue Basah (Kue Lumpur / Risoles)
- 1 Kacang Bawang / Permen
- 1 Air Mineral Gelas (240ml)

**Snack

## 5. Metadata Filtering

In [6]:
query = 'Paket untuk acara casual'
# Filter hanya tipe produk dan category 'minibox'
filters = {'type': 'product', 'category': 'minibox'}

results_filtered = rag.search(query, top_k=1, metadata_filters=filters)

print(f'Query: "{query}"')
print(f'Filters: {filters}\n')
for i, res in enumerate(results_filtered, 1):
    print(f'--- Hasil {i} (Source: {res.metadata.get("source", "")}) ---')
    print(res.metadata)
    print()

Query: "Paket untuk acara casual"
Filters: {'type': 'product', 'category': 'minibox'}

--- Hasil 1 (Source: Paket_Minibox.md) ---
{'document_id': 'P001', 'type': 'product', 'category': 'minibox', 'price': 17000, 'minimum_order': 20, 'event_types': ['gathering', 'office_break', 'casual_event'], 'active': True, 'source': 'Paket_Minibox.md', 'chunk_id': 0, 'relevance_score': 0.8264}



## 6. Context Construction untuk LLM

Bagian ini menunjukkan bagaimana output dari search dibentuk menjadi satu teks panjang untuk diberikan kepada LLM sebagai *context* (RAG).

In [7]:
context = rag.construct_context(results_filtered)
print('=== CONTEXT STRING ===\n')
print(context)
print('\n========================')

=== CONTEXT STRING ===

[Document 1 | Source: Paket_Minibox.md]
# Nasi Kotak Minibox

**Harga:** Rp 17.000 / box  
**Minimum Order:** 20 box  
**Cocok untuk:** Gathering, Office Break, Casual Event

## Menu Utama
- 1 potong ayam broiler reguler
- 1 nasi putih
- Sambal & lalapan

## Deskripsi
Nasi Kotak Minibox adalah pilihan terjangkau dari Ayam Bakar Pak D. Dengan daging ayam broiler berkualitas yang dibakar sempurna, menghadirkan cita rasa autentik Ayam Bakar Pak D dengan harga ekonomis. Cocok untuk kebutuhan sehari-hari atau acara santai.




## 7. RAG + Groq Integration Test (Section 3.8)

Mari kita integrasikan context yang didapat dengan Groq.

In [8]:
load_dotenv()

api_key = os.getenv('GROQ_API_KEY')
if api_key:
    client = Groq(api_key=api_key)
    
    # Test pertanyaan yang butuh RAG
    pertanyaan = "Berapa minimum order untuk Nasi Kotak Minibox dan apa cocok buat casual event?"
    
    # Dapatkan Context dari RAG
    rag_results = rag.search(pertanyaan, top_k=2)
    rag_context = rag.construct_context(rag_results)
    
    system_prompt = f"""Anda adalah asisten virtual Nasi Kotak.\n
Jawab pertanyaan HANYA berdasarkan konteks berikut. Jika tidak ada di konteks, bilang 'Saya tidak tahu'.\n
\n
KONTEKS:\n
{rag_context}\n
"""
    
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": pertanyaan}
        ]
    )
    print("Pertanyaan:", pertanyaan)
    print("\n--- Jawaban Groq (Dengan RAG) ---")
    print(response.choices[0].message.content)
else:
    print("GROK_API_KEY belum diset di .env")

Pertanyaan: Berapa minimum order untuk Nasi Kotak Minibox dan apa cocok buat casual event?

--- Jawaban Groq (Dengan RAG) ---
Untuk Nasi Kotak Minibox, minimum order-nya adalah 20 box.

Dan, ya, Nasi Kotak Minibox ini cocok untuk casual event!


## 8. Perbandingan RAG vs Tanpa RAG (Hallucination Test) (Section 3.9)

In [9]:
if api_key:
    pertanyaan_halu = "Apa saja menu di Nasi Kotak Broiler Jumbo dan berapa harganya?"
    
    print("\n--- 1. Tanpa RAG (Bisa Halusinasi) ---")
    response_no_rag = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {"role": "system", "content": "Kamu adalah asisten virtual Nasi Kotak."},
            {"role": "user", "content": pertanyaan_halu}
        ]
    )
    print(response_no_rag.choices[0].message.content)
    
    print("\n--- 2. Dengan RAG (Akurat) ---")
    rag_results_halu = rag.search(pertanyaan_halu, top_k=2)
    rag_context_halu = rag.construct_context(rag_results_halu)
    system_prompt_halu = f"Jawab HANYA berdasarkan konteks:\n{rag_context_halu}"
    response_with_rag = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {"role": "system", "content": system_prompt_halu},
            {"role": "user", "content": pertanyaan_halu}
        ]
    )
    print(response_with_rag.choices[0].message.content)



--- 1. Tanpa RAG (Bisa Halusinasi) ---
Berikut adalah menu dan harga dari Nasi Kotak Broiler Jumbo:

- Ayam Broiler Jumbo: Rp 35.000
    Berisi: Ayam broiler jumbo suwir, nasi putih, sambal khas, sayuran tumis, dan sambal kecap.

- Daging Sapi Broiler Jumbo: Rp 40.000
    Berisi: Daging sapi broiler jumbo suwir, nasi putih, sambal khas, sayuran tumis, dan sambal kecap.

- Kambing Broiler Jumbo: Rp 42.000
    Berisi: Kambing broiler jumbo suwir, nasi putih, sambal khas, sayuran tumis, dan sambal kecap.

- Sayuran Broiler :Rp 28.000
    Berisi: Sayuran broiler, nasi putih, sambal khas, dan sambal kecap

- Sambal Khas: Rp 8.000
- Gado-Gado: Rp 15.000

--- 2. Dengan RAG (Akurat) ---
Menu di Nasi Kotak Broiler Jumbo adalah:

1. 1 potong ayam broiler jumbo
2. 1 nasi putih
3. Tahu saus
4. Kerupuk
5. Sambal & lalapan

Harga satu box Nasi Kotak Broiler Jumbo adalah Rp 23.000.
